# Notebook 2 — Feature Engineering
Analyse et sélection des features pertinentes pour détecter les bots Twitter.
4 catégories : activité du compte, réseau social, identité, comportement temporel.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_parquet('../data/twitter_clean.parquet')
df.head()

## Catégorie 1 — Activité du compte
`statuses_count`, `favourites_count`, `listed_count`, `account_age`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['statuses_count', 'account_age']):
    df.groupby('class_bot')[col].median().plot.bar(ax=ax, color=['steelblue', 'crimson'])
    ax.set_xticklabels(['Humain', 'Bot'], rotation=0)
    ax.set_title(f'Médiane de {col}')
plt.tight_layout()
plt.show()

## Catégorie 2 — Réseau social
`followers_count`, `friends_count`, `friends_followers_ratio`, `followers_friends_ratio`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ['friends_followers_ratio', 'followers_friends_ratio']):
    df.groupby('class_bot')[col].median().plot.bar(ax=ax, color=['steelblue', 'crimson'])
    ax.set_xticklabels(['Humain', 'Bot'], rotation=0)
    ax.set_title(f'Médiane de {col}')
plt.tight_layout()
plt.show()

## Catégorie 3 — Identité du profil
`default_profile`, `default_profile_image`, `name_length`, `description_length`, `name_entropy`

In [ ]:
identity_cols = ['default_profile', 'default_profile_image', 'name_contains_bot', 'screen_name_contains_bot']
df.groupby('class_bot')[identity_cols].mean().T.plot.bar(figsize=(10, 4), color=['steelblue', 'crimson'])
plt.title('Taux moyen — features identité (0=humain, 1=bot)')
plt.xticks(rotation=30)
plt.legend(['Humain', 'Bot'])
plt.tight_layout()
plt.show()

## Catégorie 4 — Ratios temporels
`statuses_account_age_ratio`, `followers_account_age_ratio`, etc. — vitesse d'acquisition normalisée par l'âge du compte.

In [ ]:
ratio_cols = ['statuses_account_age_ratio', 'followers_account_age_ratio', 'friends_account_age_ratio']
df.groupby('class_bot')[ratio_cols].median().T.plot.bar(figsize=(10, 4), color=['steelblue', 'crimson'])
plt.title('Médiane des ratios temporels (0=humain, 1=bot)')
plt.xticks(rotation=30)
plt.legend(['Humain', 'Bot'])
plt.tight_layout()
plt.show()

## Corrélation des features avec le label

In [ ]:
FEATURES = [
    'statuses_count', 'followers_count', 'friends_count', 'favourites_count',
    'listed_count', 'default_profile', 'default_profile_image', 'geo_enabled',
    'profile_use_background_image', 'verified',
    'name_length', 'screen_name_length', 'description_length',
    'name_contains_bot', 'screen_name_contains_bot',
    'name_entropy', 'screen_name_entropy',
    'friends_followers_ratio', 'followers_friends_ratio',
    'statuses_followers_ratio', 'statuses_friends_ratio',
    'favourites_statuses_ratio', 'listed_followers_ratio',
    'account_age',
    'followers_account_age_ratio', 'friends_account_age_ratio',
    'statuses_account_age_ratio', 'favourites_account_age_ratio',
    'lists_account_age_ratio',
]

corr = df[FEATURES + ['class_bot']].corr()['class_bot'].drop('class_bot').sort_values()
plt.figure(figsize=(8, 10))
corr.plot.barh(color=['crimson' if v > 0 else 'steelblue' for v in corr])
plt.title('Corrélation avec le label (bot=1)')
plt.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.savefig('../plots/feature_correlation.png', dpi=120)
plt.show()
print(f'{len(FEATURES)} features sélectionnées')